# 02 - Train Liveness Model

This notebook trains a baseline liveness model using cropped face manifests.

- Input: `train.csv`, `val.csv` from notebook 01
- Output: best checkpoint (`best_model.pt`) and TorchScript export (`best_model_scripted.pt`)


In [ ]:
# !pip install -q -r /kaggle/working/Face_Anti_Spoofing_Biometric/requirements-kaggle.txt

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path('/kaggle/working/Face_Anti_Spoofing_Biometric')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fas.train_torch import TrainConfig, train_model

In [ ]:
PREPARED_ROOT = Path('/kaggle/working/celeba_spoof_prepared')
MANIFEST_DIR = PREPARED_ROOT / 'manifests'
OUTPUT_DIR = Path('/kaggle/working/celeba_spoof_training')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_manifest = MANIFEST_DIR / 'train.csv'
val_manifest = MANIFEST_DIR / 'val.csv'

print('Train manifest exists:', train_manifest.exists())
print('Val manifest exists:', val_manifest.exists())
print('Output dir:', OUTPUT_DIR)

In [ ]:
config = TrainConfig(
    train_manifest=str(train_manifest),
    val_manifest=str(val_manifest),
    output_dir=str(OUTPUT_DIR),
    batch_size=128,
    epochs=12,
    lr=1e-3,
    image_size=80,
    num_workers=2,
    seed=42,
)

print(config)

In [ ]:
result = train_model(config)
print('Best checkpoint:', result['best_checkpoint'])
print('Best TorchScript checkpoint:', result['best_scripted_checkpoint'])
print('Best validation accuracy:', result['best_val_acc'])

In [ ]:
history_path = OUTPUT_DIR / 'history.json'
with history_path.open('w') as fp:
    json.dump(result['history'], fp, indent=2)

print('Saved history to:', history_path)
print('Epoch metrics sample:')
print(result['history'][-1] if result['history'] else 'No history')

In [ ]:
# Save a lightweight run summary used by evaluation notebook
summary_path = OUTPUT_DIR / 'run_summary.json'
summary = {
    'best_checkpoint': result['best_checkpoint'],
    'best_scripted_checkpoint': result['best_scripted_checkpoint'],
    'best_val_acc': result['best_val_acc'],
}
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary:', summary_path)